In [ ]:
import pandas as pd
from tqdm import tqdm
import os

import sys
sys.path.append('..')
from mlrose_hiive import FlipFlopGenerator
from mlrose_hiive import MIMICRunner

In [ ]:
EXPERIMENT_NAME = 'MIMIC_FlipFlop'
SEED = 0
PROBLEM_SIZE_LIST = [1000]
ITERATIONS_LIST = [1e9]
# MAX_ATTEMPTS_LIST = [50]
MAX_ATTEMPTS_LIST = [100]
POPULATION_SIZES_LIST = [100]
# KEEP_PERCENT_LIST = [0.25, 0.5, 0.75, 0.9]
KEEP_PERCENT_LIST = [0.5]
NUM_RUNS = 5

In [ ]:
output_dir = f'metrics/{EXPERIMENT_NAME}'
os.makedirs(output_dir, exist_ok=True)

In [ ]:
all_df = pd.DataFrame()
group_i = 0
run_i = 0
for problem_size in PROBLEM_SIZE_LIST:
    for iterations in ITERATIONS_LIST:
        for max_attempts in MAX_ATTEMPTS_LIST:
            for population_size in POPULATION_SIZES_LIST:
                for keep_percent in KEEP_PERCENT_LIST:
                    for i in tqdm(range(NUM_RUNS)):
                        problem = FlipFlopGenerator().generate(seed=SEED+i, size=problem_size)
                        sa = MIMICRunner(
                            problem=problem,
                            experiment_name='dontcare',
                            output_directory='/Users/sdale/temp',
                            seed=SEED+i,
                            max_attempts=max_attempts,
                            iteration_list=[iterations],
                            population_sizes=[population_size],
                            keep_percent_list=[keep_percent],
                        )
                        _, df_run_curves = sa.run()
                        df_run_curves['group_number'] = group_i
                        df_run_curves['run_number'] = run_i
                        df_run_curves['problem_size'] = problem_size
                        df_run_curves['max_iterations'] = iterations
                        df_run_curves['max_attempts'] = max_attempts
                        df_run_curves['population_size'] = population_size
                        df_run_curves['keep_percent'] = keep_percent

                        all_df = pd.concat([all_df, df_run_curves], axis=0)
                        run_i += 1
                    group_i += 1
all_df.reset_index(inplace=True, drop=True)

In [ ]:
all_df['Fitness'].min(), all_df['Fitness'].max()

In [ ]:
all_df.to_csv(os.path.join(output_dir, 'learning_curve.csv'), index=False)